
# Smart Energy Consumption Prediction

## 1. Introduction

This notebook presents the initial inspection of the
electricity consumption dataset used in this project.

The purpose of this phase is to understand the structure,
quality, and characteristics of the data before applying
data cleaning techniques or machine learning models.

### Objectives

- Load the raw dataset.
- Identify the number of observations and variables.
- Examine data types and descriptive statistics.
- Detect missing values.
- Prepare the dataset for exploratory data analysis.

### Dataset

Source: UCI Machine Learning Repository

Dataset: Individual Household Electric Power Consumption

The dataset contains household electricity measurements
recorded over time.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "household_power_consumption.txt"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\hp\smart-energy-consumption-prediction
Dataset exists: True


This cell imports the required libraries, defines the project and dataset paths, and checks whether the dataset file exists.

In [2]:
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    na_values=["?"],
    low_memory=False
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


This cell loads the electricity consumption dataset into a Pandas DataFrame, treats `"?"` as a missing value, and displays the dataset dimensions and first five rows.

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 176.0 MB


This cell displays the DataFrame's structure, including column names, data types, non-null counts, and memory usage.

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Global_active_power,2049280.0,1.091615,1.057294,0.076,0.308,0.602,1.528,11.122
Global_reactive_power,2049280.0,0.123714,0.112722,0.000,0.048,0.100,0.194,1.390
Voltage,2049280.0,240.839858,3.239987,223.200,238.990,241.010,242.890,254.150
Global_intensity,2049280.0,4.627759,4.444396,0.200,1.400,2.600,6.400,48.400
Sub_metering_1,2049280.0,1.121923,6.153031,0.000,0.000,0.000,0.000,88.000
Sub_metering_2,2049280.0,1.298520,5.822026,0.000,0.000,0.000,1.000,80.000
Sub_metering_3,2049280.0,6.458447,8.437154,0.000,0.000,1.000,17.000,31.000


This cell generates descriptive statistics for each numerical column, including the mean, standard deviation, minimum, maximum, and quartiles, with `.T` transposing the results for easier reading.

In [17]:
missing = df.isna().sum()

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": (missing / len(df) * 100).round(2)
})

display(
    missing_report.sort_values(
        "missing_percentage",
    )
)


,missing_count,missing_percentage
Date,0,0.00
Time,0,0.00
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25


This cell calculates the number and percentage of missing values in each column, then displays the results sorted by missing-value percentage to identify columns that require data cleaning.

In [19]:
rows_with_missing = df.isna().any(axis=1).sum()
rows_without_missing = df.notna().all(axis=1).sum()

print(f"Total rows : {len(df)}")
print(f"Rows with missing values : {rows_with_missing}")
print(f"Rows without missing values : {rows_without_missing}")

print(
    "Percentage of rows with missing values :", 
    round(rows_with_missing / len(df) * 100, 2), 
    "%"
)

Total rows : 2075259
Rows with missing values : 25979
Rows without missing values : 2049280
Percentage of rows with missing values : 1.25 %


This cell counts the rows with and without missing values and calculates the percentage of rows affected by missing data.

In [20]:
missing_rows = df[df.isna().any(axis=1)]

display(
    missing_rows.isna().sum(axis=1)
    .value_counts()
    .sort_index()
)

7    25979
Name: count, dtype: int64

This cell examines rows containing missing values and counts how many rows have each number of missing entries.

In [21]:
print("Date data type:", df["Date"].dtype)
print("Time data type:", df["Time"].dtype)

print("First date:", df["Date"].iloc[0])
print("Last date:", df["Date"].iloc[-1])

print("Number of unique dates:", df["Date"].nunique())

Date data type: str
Time data type: str
First date: 16/12/2006
Last date: 26/11/2010
Number of unique dates: 1442


This cell checks the data types of the date and time columns, identifies the first and last dates, and counts the number of unique dates in the dataset.

In [22]:
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

print("Datetime data type:", df["Datetime"].dtype)
display(df[["Date", "Time", "Datetime"]].head())

Datetime data type: datetime64[us]


,Date,Time,Datetime
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00


This cell combines the date and time columns into a single datetime column, converting the values into a format suitable for time-series analysis.

In [23]:
print(
    "Data is chronologically sorted:",
    df["Datetime"].is_monotonic_increasing
)

print(
    "Number of duplicated timestamps:",
    df["Datetime"].duplicated().sum()
)

print(
    "Minimum timestamp:",
    df["Datetime"].min()
)

print(
    "Maximum timestamp:",
    df["Datetime"].max()
)

Data is chronologically sorted: True
Number of duplicated timestamps: 0
Minimum timestamp: 2006-12-16 17:24:00
Maximum timestamp: 2010-11-26 21:02:00


This cell verifies the chronological order of the dataset, counts duplicated timestamps, and identifies the earliest and latest observations.

In [24]:
numeric_columns = df.select_dtypes(include="number").columns

display(df[numeric_columns].describe().T)

print("Negative values per column:")
display((df[numeric_columns] < 0).sum())

,count,mean,std,min,25%,50%,75%,max
Global_active_power,2049280.0,1.091615,1.057294,0.076,0.308,0.602,1.528,11.122
Global_reactive_power,2049280.0,0.123714,0.112722,0.000,0.048,0.100,0.194,1.390
Voltage,2049280.0,240.839858,3.239987,223.200,238.990,241.010,242.890,254.150
Global_intensity,2049280.0,4.627759,4.444396,0.200,1.400,2.600,6.400,48.400
Sub_metering_1,2049280.0,1.121923,6.153031,0.000,0.000,0.000,0.000,88.000
Sub_metering_2,2049280.0,1.298520,5.822026,0.000,0.000,0.000,1.000,80.000
Sub_metering_3,2049280.0,6.458447,8.437154,0.000,0.000,1.000,17.000,31.000


Negative values per column:


Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

This cell summarizes the numerical variables and counts negative values in each column to identify potentially invalid observations.

In [25]:
inspection_summary = {
    "total_rows": len(df),
    "total_columns": df.shape[1],
    "missing_values": int(df.isna().sum().sum()),
    "duplicated_timestamps": int(df["Datetime"].duplicated().sum()),
    "negative_numeric_values": int((df[numeric_columns] < 0).sum().sum()),
    "datetime_sorted": bool(df["Datetime"].is_monotonic_increasing)
}

display(pd.Series(inspection_summary, name="Value"))

total_rows                 2075259
total_columns                   10
missing_values              181853
duplicated_timestamps            0
negative_numeric_values          0
datetime_sorted               True
Name: Value, dtype: object

This cell summarizes the dataset inspection results, including its dimensions, missing values, duplicated timestamps, negative numerical values, and chronological order.

## Conclusion

The dataset contains 2,075,259 rows and 10 columns, including the combined `Datetime` column. The timestamps are chronologically ordered, with no duplicated timestamps or negative numerical values detected. However, 181,853 missing values remain across the seven numerical measurement columns. These missing values will be investigated and handled during the data cleaning stage.
